# Pràctica 1 – El cas Airbnb: càrrega, neteja i exploració base
**Ciutats:** Milan, Mallorca, Munic (Munich), Columbus

Aquest notebook és la **base comuna** del grup: carrega els `listings.csv.gz` de les 4 ciutats, revisa l'estructura, neteja les dades i desa un dataset net que faran servir tots els estudis (Part 1) i els mapes (Part 2).

**Estructura de carpetes esperada**
```
data/
  milan/     listings.csv.gz   neighbourhoods.geojson
  mallorca/  listings.csv.gz   neighbourhoods.geojson
  munich/    listings.csv.gz   neighbourhoods.geojson
  columbus/  listings.csv.gz   neighbourhoods.geojson
  clean/     (es genera automàticament)
```

## 0. Imports i configuració

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

DATA = Path('data')
CITIES = {'Milan': 'milan', 'Mallorca': 'mallorca', 'Munich': 'munich', 'Columbus': 'columbus'}
CURRENCY = {'Milan': 'EUR', 'Mallorca': 'EUR', 'Munich': 'EUR', 'Columbus': 'USD'}

## 1. Càrrega de les dades

In [ ]:
raw = {}
for city, folder in CITIES.items():
    df = pd.read_csv(DATA / folder / 'listings.csv.gz', low_memory=False)
    df['city'] = city
    raw[city] = df
    print(f'{city:10s} -> {df.shape[0]:>6} allotjaments, {df.shape[1]} variables')

## 2. Estructura i contingut de les dades
Comprovem que les 4 ciutats tenen les mateixes columnes i quins tipus de dades tenen (a `raw` tot és encara sense netejar: `price` és text, les dates són text, etc.).

In [ ]:
cols = {c: set(df.columns) for c, df in raw.items()}
common = set.intersection(*cols.values())
print('Columnes comunes a les 4 ciutats:', len(common))
for c, s in cols.items():
    extra = s - common
    if extra:
        print(f'Només a {c}:', sorted(extra))

first = raw['Milan']
pd.DataFrame({'dtype': first.dtypes.astype(str),
              'exemple': first.iloc[0],
              'nuls_%': (first.isna().mean() * 100).round(1)})

### 2.1 Selecció de variables
Ens quedem amb les variables útils per als 9 estudis i els mapes (característiques, ubicació, propietari, disponibilitat i ressenyes).

In [ ]:
KEEP = [
    'id', 'city', 'name',
    # allotjament
    'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'bathrooms_text', 'amenities',
    # ubicació
    'latitude', 'longitude', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    # propietari
    'host_id', 'host_since', 'host_is_superhost', 'host_response_time', 'host_response_rate',
    'host_listings_count', 'calculated_host_listings_count', 'host_identity_verified',
    # preu i disponibilitat
    'price', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_365', 'instant_bookable',
    # ressenyes
    'number_of_reviews', 'number_of_reviews_ltm', 'first_review', 'last_review', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value',
]
missing_cols = [c for c in KEEP if c not in common]
print('Columnes no disponibles a alguna ciutat (s\'ignoren):', missing_cols)
KEEP = [c for c in KEEP if c in common]

## 3. Neteja
Transformacions:
- `price`: text (`"$1,200.00"`) → número.
- `host_is_superhost`, `instant_bookable`, `host_identity_verified`: `t`/`f` → booleà.
- `host_response_rate`: `"95%"` → 95.0.
- Dates (`host_since`, `first_review`, `last_review`) → `datetime`.
- `bathrooms`: si és nul, es dedueix de `bathrooms_text` (`"Half-bath"` → 0.5).
- `amenities`: llista JSON → `n_amenities` (nombre de serveis) i `amenities_list`.
- Duplicats per `id` eliminats.

In [ ]:
def clean_listings(df):
    df = df[KEEP].copy()
    df = df.drop_duplicates(subset='id')

    # Preu: text -> float
    df['price'] = pd.to_numeric(
        df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    df['currency'] = CURRENCY[df['city'].iloc[0]]

    # Booleans t/f
    for c in ['host_is_superhost', 'instant_bookable', 'host_identity_verified']:
        if c in df:
            df[c] = df[c].map({'t': True, 'f': False})

    # Percentatge de resposta
    if 'host_response_rate' in df:
        df['host_response_rate'] = pd.to_numeric(
            df['host_response_rate'].astype(str).str.replace('%', '', regex=False), errors='coerce')

    # Dates
    for c in ['host_since', 'first_review', 'last_review']:
        if c in df:
            df[c] = pd.to_datetime(df[c], errors='coerce')

    # Banys
    if 'bathrooms_text' in df:
        txt = df['bathrooms_text'].astype(str)
        num = pd.to_numeric(txt.str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
        half = txt.str.contains('half', case=False, na=False) & num.isna()
        num = num.mask(half, 0.5)
        df['bathrooms'] = df['bathrooms'].fillna(num)

    # Serveis
    def parse_amen(x):
        try:
            return json.loads(x)
        except Exception:
            return []
    df['amenities_list'] = df['amenities'].apply(parse_amen)
    df['n_amenities'] = df['amenities_list'].str.len()
    df = df.drop(columns=['amenities'])
    return df

clean = {city: clean_listings(df) for city, df in raw.items()}
for city, df in clean.items():
    print(f'{city:10s} {df.shape[0]:>6} files (abans {raw[city].shape[0]})')

## 4. Valors desconeguts (nuls)
Mirem el % de nuls per variable i ciutat. **Criteri:** no s'imputa el preu (sense preu l'allotjament no serveix per als estudis de preu); els nuls de `review_scores_*` són allotjaments **sense ressenyes** i es deixen com a NaN; `bedrooms`/`beds` es deixen NaN i cada estudi decideix si els descarta.

In [ ]:
nulls = pd.DataFrame({c: (d.isna().mean() * 100).round(1) for c, d in clean.items()})
nulls[(nulls > 0).any(axis=1)].sort_values(by='Milan', ascending=False)

In [ ]:
# Preus nuls o 0 -> no vàlids
for city, d in clean.items():
    bad = d['price'].isna() | (d['price'] <= 0)
    print(f'{city:10s} preu nul o <=0: {bad.sum()} ({bad.mean()*100:.1f}%)')

valid = pd.concat(
    [d[d['price'].notna() & (d['price'] > 0)] for d in clean.values()], ignore_index=True)
print('\nTotal allotjaments amb preu vàlid:', len(valid))

## 5. Valors atípics (outliers)
Mètode: **regla de l'IQR per ciutat** (`Q1 − 1.5·IQR`, `Q3 + 1.5·IQR`) sobre `price`, ja que les ciutats tenen escales de preu diferents. Marquem els outliers amb una columna i **no els esborrem** de `valid`: així cada estudi pot triar. Generem també `df_clean` (sense outliers de preu ni estades mínimes absurdes) per a la majoria d'anàlisis.

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

valid['price_outlier'] = False
rows = []
for city in CITIES:
    m = valid['city'] == city
    lo, hi = iqr_bounds(valid.loc[m, 'price'])
    out = m & ((valid['price'] < lo) | (valid['price'] > hi))
    valid.loc[out, 'price_outlier'] = True
    rows.append({'ciutat': city, 'límit_inf': round(lo, 1), 'límit_sup': round(hi, 1),
                 'n_outliers': int(out.sum()), '% outliers': round(out.sum() / m.sum() * 100, 1),
                 'preu_max': valid.loc[m, 'price'].max()})
pd.DataFrame(rows)

In [ ]:
# Estades mínimes absurdes (> 1 any) marcades també
valid['minnights_outlier'] = valid['minimum_nights'] > 365

df_clean = valid[~valid['price_outlier'] & ~valid['minnights_outlier']].copy()
print('valid:', len(valid), '| df_clean:', len(df_clean))
df_clean.groupby('city').size()

## 6. Exploració bàsica (visió general)

### 6.1 Nombre d'allotjaments i tipus d'habitació per ciutat

In [ ]:
print(valid.groupby('city').size().rename('allotjaments'))
pd.crosstab(valid['city'], valid['room_type'], normalize='index').round(3) * 100

### 6.2 Preu: estadístics descriptius per ciutat (sense outliers)
Les monedes són diferents (Columbus en USD), per tant es comparen sobretot formes de distribució (CV, asimetria, curtosi).

In [ ]:
def describe_price(s):
    return pd.Series({
        'n': s.size, 'mitjana': s.mean(), 'mediana': s.median(),
        'Q1': s.quantile(.25), 'Q3': s.quantile(.75), 'IQR': s.quantile(.75) - s.quantile(.25),
        'std': s.std(), 'CV': s.std() / s.mean(),
        'asimetria': stats.skew(s), 'curtosi': stats.kurtosis(s)})

df_clean.groupby('city')['price'].apply(describe_price).unstack().round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for city in CITIES:
    sns.histplot(df_clean.loc[df_clean.city == city, 'price'], bins=40, stat='density',
                 element='step', fill=False, label=city, ax=ax[0])
ax[0].set_title('Distribució del preu per nit'); ax[0].legend()

sns.boxplot(data=df_clean, x='city', y='price', ax=ax[1])
ax[1].set_title('Boxplot del preu (sense outliers)')

sns.boxplot(data=valid.assign(log_price=np.log1p(valid['price'])), x='city', y='log_price', ax=ax[2])
ax[2].set_title('Boxplot de log(preu) (amb tots els valors)')
plt.tight_layout(); plt.show()

### 6.3 Altres variables numèriques clau

In [ ]:
num_vars = ['accommodates', 'bedrooms', 'bathrooms', 'n_amenities', 'minimum_nights',
            'availability_365', 'number_of_reviews', 'review_scores_rating']
df_clean.groupby('city')[num_vars].median().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for a, v in zip(axes.ravel(), num_vars):
    sns.boxplot(data=df_clean, x='city', y=v, ax=a, showfliers=False)
    a.set_title(v); a.set_xlabel('')
plt.tight_layout(); plt.show()

## 7. Comprovació dels GeoJSON
Verifiquem que el nom de barri de `listings` (`neighbourhood_cleansed`) coincideix amb el del GeoJSON, necessari per als mapes de coropletes.

In [ ]:
geo = {}
for city, folder in CITIES.items():
    with open(DATA / folder / 'neighbourhoods.geojson', encoding='utf-8') as f:
        geo[city] = json.load(f)
    names_geo = {ft['properties']['neighbourhood'] for ft in geo[city]['features']}
    names_lst = set(df_clean.loc[df_clean.city == city, 'neighbourhood_cleansed'].dropna())
    print(f'{city:10s} barris GeoJSON: {len(names_geo):>3} | barris a listings: {len(names_lst):>3} '
          f'| sense coincidència: {len(names_lst - names_geo)}')

## 8. Desar el dataset net
Tots els estudis i mapes parteixen d'aquests fitxers.

In [ ]:
out = DATA / 'clean'
out.mkdir(exist_ok=True)

# amenities_list es desa com a text JSON
to_save = valid.copy()
to_save['amenities_list'] = to_save['amenities_list'].apply(json.dumps)
to_save.to_csv(out / 'listings_valid.csv.gz', index=False)
to_save[to_save.index.isin(df_clean.index)].to_csv(out / 'listings_clean.csv.gz', index=False)
print('Desat a', out.resolve())

### Com carregar-lo als estudis individuals
```python
import pandas as pd, json
df = pd.read_csv('data/clean/listings_clean.csv.gz',
                 parse_dates=['host_since', 'first_review', 'last_review'])
df['amenities_list'] = df['amenities_list'].apply(json.loads)
```
- `listings_clean.csv.gz`: sense outliers (per defecte).
- `listings_valid.csv.gz`: amb outliers marcats (`price_outlier`, `minnights_outlier`), per si un estudi els necessita (p. ex. dominància de grans propietaris).